In [1]:
 # Let’s create a tool that queries a vector database of technical documentation or specialized knowledge. 
# Using semantic search, the agent can find the most relevant information for Alfred’s needs.

In [2]:
# create a tool that retrieves party planning ideas from a custom knowledge base. 
# We’ll use a BM25 retriever to search the knowledge base and return the top results, and 
# RecursiveCharacterTextSplitter to split the documents into smaller chunks for more efficient search.

In [5]:
from langchain_community.docstore.document import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from smolagents import Tool
from langchain_community.retrievers import BM25Retriever
from smolagents import CodeAgent, InferenceClientModel

In [6]:
class PartyPlanningRetrieverTool(Tool):
    name = "party_planning_retriever"
    description = "Uses semantic search to retrieve relevant party planning ideas for Alfred’s superhero-themed party at Wayne Manor."
    inputs = {
        "query": {
            "type": "string",
            "description": "The query to perform. This should be a query related to party planning or superhero themes.",
        }
    }
    output_type = "string"

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        self.retriever = BM25Retriever.from_documents(
            docs, k=5  # Retrieve the top 5 documents
        )

    def forward(self, query: str) -> str:
        assert isinstance(query, str), "Your search query must be a string"

        docs = self.retriever.invoke(
            query,
        )
        return "\nRetrieved ideas:\n" + "".join(
            [
                f"\n\n===== Idea {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )

In [9]:
# Simulate a knowledge base about party planning
party_ideas = [
    {"text": "A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.", "source": "Party Ideas 1"},
    {"text": "Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.", "source": "Entertainment Ideas"},
    {"text": "For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'", "source": "Catering Ideas"},
    {"text": "Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.", "source": "Decoration Ideas"},
    {"text": "Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.", "source": "Entertainment Ideas"}
]

In [10]:
source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"]})
    for doc in party_ideas
]

In [11]:
source_docs

[Document(metadata={'source': 'Party Ideas 1'}, page_content='A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.'),
 Document(metadata={'source': 'Entertainment Ideas'}, page_content='Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.'),
 Document(metadata={'source': 'Catering Ideas'}, page_content="For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'"),
 Document(metadata={'source': 'Decoration Ideas'}, page_content='Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.'),
 Document(metadata={'source': 'Entertainment Ideas'}, page_content='Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.')]

In [26]:
# Split the documents into smaller chunks for more efficient search
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ".", " ", ""],
)
docs_processed = text_splitter.split_documents(source_docs)

In [27]:
docs_processed

[Document(metadata={'source': 'Party Ideas 1', 'start_index': 0}, page_content='A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.'),
 Document(metadata={'source': 'Entertainment Ideas', 'start_index': 0}, page_content='Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.'),
 Document(metadata={'source': 'Catering Ideas', 'start_index': 0}, page_content="For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'"),
 Document(metadata={'source': 'Decoration Ideas', 'start_index': 0}, page_content='Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.'),
 Document(metadata={'source': 'Entertainment Ideas', 'start_index': 0}, page_content='Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.')]

In [28]:
# Create the retriever tool
party_planning_retriever = PartyPlanningRetrieverTool(docs_processed)

In [29]:
# Initialize the agent
agent = CodeAgent(tools=[party_planning_retriever], model=InferenceClientModel())

In [30]:
# Example usage
response = agent.run(
    "Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration options."
)

print(response)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find ideas for a luxury superhero-themed party, including entertainment, catering, and decoration options.      │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Next-80B-A3B-Thinking ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  query = "luxury superhero-themed party for Wayne Manor entertainment catering decoration"                        
  ideas = party_planning_retriever(query=query)                                                                    
  print(ideas)                                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:

Retrieved ideas:


===== Idea 0 =====
A superhero-themed masquerade ball with luxury decor, including gold accents and velvet curtains.

===== Idea 1 =====
Hire a professional DJ who can play themed music for superheroes like Batman and Wonder Woman.

===== Idea 2 =====
Interactive experiences with VR where guests can engage in superhero simulations or compete in themed games.

===== Idea 3 =====
Decorate with iconic superhero logos and projections of Gotham and other superhero cities around the venue.

===== Idea 4 =====
For catering, serve dishes named after superheroes, like 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak.'

Out: None

[Step 1: Duration 12.17 seconds| Input tokens: 2,103 | Output tokens: 2,267]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Decoration: Gold accents, velvet curtains, iconic superhero logos, and projections of Gotham       
  City.\nEntertainment: Professional DJ playing superhero-themed music and interactive VR superhero                
  simulations.\nCatering: Dishes named after superheroes such as 'The Hulk's Green Smoothie' and 'Iron Man's       
  Power Steak'.")                                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Decoration: Gold accents, velvet curtains, iconic superhero logos, and projections of Gotham City.
Entertainment: Professional DJ playing superhero-themed music and interactive VR superhero simulations.
Catering: Dishes named after superheroes such as 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak'.

[Step 2: Duration 15.25 seconds| Input tokens: 4,536 | Output tokens: 5,347]

Decoration: Gold accents, velvet curtains, iconic superhero logos, and projections of Gotham City.
Entertainment: Professional DJ playing superhero-themed music and interactive VR superhero simulations.
Catering: Dishes named after superheroes such as 'The Hulk's Green Smoothie' and 'Iron Man's Power Steak'.
